In [1]:
import fitz  # PyMuPDF
import re
import json
from typing import List, Dict, Optional
from dataclasses import dataclass, asdict
from pathlib import Path
import pandas as pd
from dataclasses import asdict

# LangChain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [2]:
restaurant_list = [
    # 국밥
    "포항돼지국밥", "송정3대국밥", "형제돼지국밥", "60년전통할매국밥", 
    "서진섭돼지국밥", "신창국밥", "밀양국밥", "자매국밥",

    # 밀면
    "내호냉면", "대성밀냉면", "개금밀면", "할매가야밀면", 

    # 복국
    "영주동삼대복국집", "할매복집", "초원복국", 
    "새송도복국", "금수복국", "남해복국", 
    "박옥희할매집원조복국", "영주복국", "가마솥생복집", "일광대복집", 

    # 대구탕
    "기와집대구탕", "오뚜기식당", "중앙식당", "속씨원한대구탕",

    # 차이나타운
    "마가만두 馬家", "신발원 新發園", "옥생관 玉生館", "동화반점 東華飯店",
    "신흥관 新興館", "신흥반점 新興飯店", "삼생원 三生園", "홍성방 鴻聖坊",
   
   # 곰장어
   "대한명인기장곰장어",
   "성일집", "원조소문난산곰장어", "온천입구기장곰장어", 

   # 멸치
   "용암할매횟집", "남항횟집", "장군멸치회촌", 

   # 생선회
   "부산명물횟집", "꺼먹동네", "제일장횟집", "무진장횟집", 
   "영변횟집", "용광횟집",

   # 해물탕
   "궁중해물탕조씨집", "원조바다집",

   # 돼지갈비
   "산수갑산", "삼대돼지불고기", "은하갈비",

   # 구포구수
    "구포촌국수", "이원화구포국시",

    # 그 외 노포들
    "평산옥", "동래할매파전", "서울깍두기", "18번완당집",
    "원조18번완당발국수", "급행장 急行莊", "새진주식당", "옛날오막집",
    "구포집", "백화양곱창", "구조방낙지",
    "태화육개장", "해운대암소갈비집", "원조전복죽집",
    "물꽁식당", "고갈비남마담", "할매재첩국",
]

In [3]:
@dataclass
class Restaurant:
    """식당 정보를 담는 데이터 클래스"""
    name_ko: str
    name_en: str
    category: str
    description: str
    menu: str
    established: str
    hours: str
    address: str
    phone: str
    page_num: int = 0
    source: str = "블루리본"
    year: int = 2025  # 연도 필드 추가

    def to_dict(self):
        return asdict(self)
    
    def to_text(self):
        """RAG용 텍스트 형식으로 변환"""
        return f"""
출처: {self.source} ({self.year}년)
식당명: {self.name_ko} ({self.name_en})
카테고리: {self.category}
설명: {self.description}
메뉴: {self.menu}
개업: {self.established}
영업시간: {self.hours}
주소: {self.address}
전화: {self.phone}
""".strip()
    
    def to_document(self) -> Document:
        """LangChain Document 형식으로 변환"""
        return Document(
            page_content=self.to_text(),
            metadata={
                "name": self.name_ko,
                "category": self.category,
                "address": self.address,
                "phone": self.phone,
                "page": self.page_num,
                "source": self.source,
                "year": self.year,  # 메타데이터에도 연도 추가
                "type": "restaurant"
            }
        )

In [20]:
class BusanNopoRAG:
    """부산 노포 RAG 시스템"""
    
    def __init__(self, use_vectordb: bool = True):
        self.restaurants: List[Restaurant] = []
        self.documents: List[Document] = []
        self.vectorstore = None
        self.use_vectordb = use_vectordb
        
        # 카테고리 매핑
        self.category_keywords = {
            '돼지국밥': ['Dwaeji-gukbap', 'Pork and Rice Soup', '국밥'],
            '밀면': ['Milmyeon', 'Cold Wheat Noodles', '냉면'],
            '복국': ['Bokguk', 'Puffer Fish Soup', '복어'],
            '대구탕': ['Daegu-tang', 'Codfish', '대구'],
            '차이나타운': ['Chinatown', '중식', '만두'],
            '곰장어': ['Gomjangeo', 'Hagfish', '장어'],
            '멸치': ['Myeolchi', 'anchovy'],
            '생선회': ['Hoe', 'raw fish', '회'],
            '해물탕': ['Haemultang', 'seafood stew'],
            '돼지갈비': ['Dwaeji Galbi', 'pork ribs', '갈비'],
            '구포국수': ['Gupo Guksu', 'wheat noodles', '국수'],
            '기타': ['Other', 'heritage']
        }
    
    def read_pdf_with_pymupdf(self, pdf_path: str) -> List[Dict]:
        """PyMuPDF를 사용하여 PDF 읽기 (페이지별)"""
        try:
            doc = fitz.open(pdf_path)
            pages_data = []
            
            for page_num in range(len(doc)):
                page = doc[page_num]
                rect = page.rect
                mid_x = rect.width / 2

                left_clip = fitz.Rect(0, 0, mid_x, rect.height)
                left_text = page.get_text("text", clip=left_clip)

                right_clip = fitz.Rect(mid_x, 0, rect.width, rect.height)
                right_text = page.get_text("text", clip=right_clip)

                full_text = f"{left_text}\n{right_text}"

                pages_data.append({
                    'page_num': page_num + 1,
                    'text': full_text
                })
            
            doc.close()
            print(f"✅ PDF 읽기 완료: 총 {len(pages_data)} 페이지")
            return pages_data
            
        except Exception as e:
            print(f"❌ PDF 읽기 실패: {e}")
            return []
        
    def read_pdf_with_pymupdf_v1(self, pdf_path: str) -> List[Dict]:
        try:
            doc = fitz.open(pdf_path)
            pages_data = []
            
            only_right_pages = [5, 7, 8, 10, 11, 13, 17, 18, 19, 20]
            only_left_pages = [14, 15]

            for page_num in range(5, len(doc)-5):
                page = doc[page_num]
                rect = page.rect
                mid_x = rect.width / 2
                # 하단 페이지 번호 영역(약 50px)을 제외하기 위한 높이 설정
                content_height = rect.height - 50 

                # 좌/우 영역 정의 (하단 제외)
                if page_num in only_right_pages:
                    clips = [
                        fitz.Rect(mid_x, 0, rect.width, content_height) # 오른쪽
                    ]
                
                elif page_num in only_left_pages:
                    clips = [
                        fitz.Rect(0, 0, mid_x, content_height),      # 왼쪽
                    ]
                else:
                    clips = [
                        fitz.Rect(0, 0, mid_x, content_height),      # 왼쪽
                        fitz.Rect(mid_x, 0, rect.width, content_height) # 오른쪽
                    ]

                full_text = []
                for clip in clips:
                    # "dict" 형식으로 읽어야 글자 색상(color) 정보가 포함됨
                    blocks = page.get_text("dict", clip=clip)["blocks"]
                    
                    for b in blocks:
                        if "lines" in b:
                            for l in b["lines"]:
                                for s in l["spans"]:
                                    # 흰색 글씨(color 16777215) 제외
                                    # PyMuPDF에서 흰색은 보통 0xFFFFFF (16777215)입니다.
                                    if s["color"] != 16777215:
                                        full_text.append(s["text"])

                pages_data.append({
                    'page_num': page_num + 1,
                    'text': "\n".join(full_text).strip()
                })
            
            doc.close()
            print(f"✅ PDF 읽기 완료: 총 {len(pages_data)} 페이지")
            return pages_data
        except Exception as e:
            print(f"❌ 오류 발생: {e}")
            return []
    
    def detect_category(self, text: str) -> str:
        """텍스트에서 카테고리 감지"""
        for category, keywords in self.category_keywords.items():
            for keyword in keywords:
                if keyword in text:
                    return category
        return '기타'
    
    def parse_restaurant_info(self, text_block: str) -> Optional[Restaurant]:
        """텍스트 블록에서 식당 정보 추출"""
        lines = [line.strip() for line in text_block.strip().split('\n') if line.strip()]
        
        if len(lines) < 2:
            return None

        # 1. 영문 이름: 첫 번째 줄 (영문과 공백으로만 이루어졌는지 확인)
        name_en = lines[0]
        if not re.search(r'[A-Za-z]', name_en): # 영문이 아예 없으면 스킵
            return None

        # 2. 한글 이름: 두 번째 줄
        name_ko = lines[1]
        description_start_idx = 2

        if len(lines[2]) < 5:
            name_ko = name_ko + " " + lines[2]
            description_start_idx = 3

        # # 한글이 포함되어 있는지 검사
        # if not re.search(r'[가-힣]', name_ko):
        #     # 만약 두 번째 줄도 영문이라면, 한글 이름은 세 번째 줄일 수 있음
        #     if len(lines) > 2 and re.search(r'[가-힣]', lines[2]):
        #         name_ko = lines[2]
        #         description_start_idx = 3
        #     else:
        #         return None
        # else:
        #     description_start_idx = 2

        # 3. 설명: 한글 이름 이후부터 키워드 등장 전까지 합치기
        description_parts = []
        keywords = ['메뉴', '개업', '영업시간', '주소', '전화']
        
        for line in lines[description_start_idx:]:
            if any(line.startswith(kw) for kw in keywords):
                break
            description_parts.append(line)
        description = "".join(description_parts)

        # 4. 나머지 정보 추출 (패턴 유연화)
        def quick_search(pattern, text, dotall=False):
            flags = re.DOTALL if dotall else 0
            match = re.search(pattern, text, flags)
            return match.group(1).strip() if match else ""

        # 메뉴: '메뉴' 단어 이후부터 다른 키워드 전까지
        menu = quick_search(r'메뉴\s*(.*?)(?=개업|영업시간|주소|전화|$)', text_block, True)
        menu = menu.replace('\n', '')
        
        # 개업년도: 숫자 4자리만 있으면 추출
        established = quick_search(r'개업\s*(\d{4}년?|\d{2}년대?)', text_block)
        established = established.replace('\n', '')

        # 영업시간: '영업시간' 이후부터 주소 전까지
        hours = quick_search(r'영업시간\s*(.*?)(?=주소|전화|$)', text_block, True)
        hours = hours.replace('\n', '')

        # 주소: '주소' 이후부터 전화 전까지
        address = quick_search(r'주소\s*(.*?)(?=전화|$)', text_block, True)
        address = address.replace('\n', '')

        # 전화번호: 숫자와 하이픈 조합
        phone = quick_search(r'전화\s*(.*?)(?=$)', text_block, True)   
        phone = phone.replace('\n', '')

        category = self.detect_category(text_block)

        return Restaurant(
            name_ko=name_ko,
            name_en=name_en,
            category=category,
            description=description,
            menu=menu,
            established=established,
            hours=hours,
            address=address,
            phone=phone,
        )
    
    def parse_pdf_pages(self, full_text: str):
        """PDF 페이지들을 파싱하여 식당 정보 추출"""
        # 1. 분할 기준 완화: 줄바꿈 뒤에 영문자가 오는 지점을 모두 후보로 잡음
        # (?!메뉴|주소|영업) 등을 넣어 주요 키워드는 분리 기준에서 제외
        # blocks = re.split(r'\n(?=[A-Za-z]{2,}(?:\s+[A-Za-z]+)*\n)', text)
        # 하이픈(-)을 허용하도록 수정된 정규식
        blocks = re.split(r'\n(?=(?=[^ \n]*[A-Za-z])[A-Za-z0-9\-]{2,}(?:\s+[A-Za-z0-9\-]+)*\n)', full_text)
        
        for block in blocks:
            # 블록이 너무 짧으면 패스 (최소 길이 조정)
            if len(block.strip()) < 20: 
                continue
            
            restaurant = self.parse_restaurant_info(block)
            if restaurant:
                self.restaurants.append(restaurant)
                print(f"✅ 파싱 완료: {restaurant.name_ko}")
                    
        return self.restaurants
    
    def create_vector_store(self, embedding_model: str = "jhgan/ko-sbert-nli"):
        """벡터 스토어 생성 (ChromaDB + Korean SBERT)"""
        
        if not self.restaurants:
            print("❌ 파싱된 식당 정보가 없습니다.")
            return
        
        # LangChain Document 생성
        self.documents = [r.to_document() for r in self.restaurants]
        
        # 한국어 임베딩 모델 초기화
        embeddings = HuggingFaceEmbeddings(
            model_name=embedding_model,
            model_kwargs={'device': 'cpu'},
            encode_kwargs={'normalize_embedings': True}
        )
        
        # ChromaDB 벡터 스토어 생성
        self.vectorstore = Chroma.from_documents(
            documents=self.documents,
            embedding=embeddings,
            persist_directory="./chroma_db",
            collection_name="busan_nopo"
        )
        
        print(f"✅ 벡터 스토어 생성 완료: {len(self.documents)}개 문서")
    
    def semantic_search(self, query: str, k: int = 3) -> List[Dict]:
        """의미 기반 검색"""
        if not self.vectorstore:
            print("❌ 벡터 스토어가 초기화되지 않았습니다.")
            return []
        
        results = self.vectorstore.similarity_search(query, k=k)
        
        return [
            {
                'content': doc.page_content,
                'metadata': doc.metadata
            }
            for doc in results
        ]
    
    def search_by_category(self, category: str) -> List[Restaurant]:
        """카테고리별 검색"""
        return [r for r in self.restaurants if r.category == category]
    
    def search_by_name(self, name: str) -> List[Restaurant]:
        """이름으로 검색"""
        return [r for r in self.restaurants 
                if name.lower() in r.name_ko.lower() or name.lower() in r.name_en.lower()]
    
    def get_all_categories(self) -> List[str]:
        """모든 카테고리 목록"""
        return list(set(r.category for r in self.restaurants))
    
    def save_to_json(self, filename: str = 'busan_nopo.json'):
        """JSON 파일로 저장"""
        data = [r.to_dict() for r in self.restaurants]
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        print(f"✅ {len(data)}개 식당 정보를 {filename}에 저장했습니다.")
    
    def save_to_text(self, filename: str = 'busan_nopo_rag.txt'):
        """텍스트 파일로 저장 (RAG용)"""
        with open(filename, 'w', encoding='utf-8') as f:
            for restaurant in self.restaurants:
                f.write(restaurant.to_text())
                f.write('\n' + '='*80 + '\n\n')
        print(f"✅ {len(self.restaurants)}개 식당 정보를 {filename}에 저장했습니다.")


In [21]:
rag = BusanNopoRAG(use_vectordb=True)

pdf_path = "./busan_restaurants_pdfs/busan_blueribbon_2025.pdf"

if Path(pdf_path).exists():
    print(f"📖 {pdf_path} 파일 읽는 중...\n")
    
    # PyMuPDF로 PDF 읽기
    pages_data = rag.read_pdf_with_pymupdf_v1(pdf_path)

📖 ./busan_restaurants_pdfs/busan_blueribbon_2025.pdf 파일 읽는 중...

✅ PDF 읽기 완료: 총 18 페이지


In [22]:
pages_data[6]

{'page_num': 12,
 'text': 'Maga Mandu\n마가만두 \n馬家\n부산 차이나타운에서 손꼽히는 만두 전문점. 얇\n은 만두피 안에 소를 꽉 채워 숙성한 뒤 쪄낸다. \n고기의 풍미와 생강 향이 좋고 가득한 육즙과 쫄\n깃한 만두피가 인상적이다.\n메뉴 \n물만두, 찐만두, 군만두, 볶음밥(각 8천원), 짬뽕밥, \n잡채밥(각 8천원), 유산슬(3만5천원), 탕수육(2만6천원), \n라조육, 깐풍육, 라조기(각 3만원)\n개업 \n1951년\n영업시간 \n11:00~21:00 - 첫째, 셋째 주 월요일 휴무 \n주소 \n부산광역시 동구 대영로243번길 56 (초량동)\n전화 \n051-468-4059  \nSinbalwon\n신발원 \n新發園\n두꺼우면서도 부드러운 만두피와 생강과 돼지고\n기를 넣은 속이 조화를 이루는 중국만두로 유명\n하다. 중국만두 외에도 팥빵, 달걀빵 등 중국빵을 \n전문으로 한다. 중국 사람들이 아침에 즐겨 찾는 \n콩국은 과자와 함께 나온다. 70년이 넘는 역사를 \n자랑하는 곳.\n메뉴 \n고기만두, 찐교자(각 4천5백원), 군만두(5천3백원), \n마라만두, 새우교자(각 6천5백원), 콩국&과자(3천5백원), \n백새우만두(4천원), 마라비빔면, 쿵푸면(각 4천5백원), \n숙주나물(2천5백원), 부추당면(5천원), 모둠만두(1만5천5\n백원), 군맥세트(1만4천3백원)\n개업 \n1951년\n영업시간 \n11:00~21:00(마지막 주문 20:20) - 화요일 휴\n무 \n주소 \n부산광역시 동구 대영로243번길 62 (초량동)\n전화 \n051-467-0177  \nOk Saeng Gwan\n옥생관 \n玉生館\n우동이 맛있기로 유명한 중식당. 짜장면과 우동\n이 인기 메뉴로, 푸짐한 해산물과 시원한 국물이 \n일품인 우동 맛이 좋다. 여름에는 중국식 냉면도 \n즐길 수 있다. 달걀프라이가 올라가는 부산식 간\n짜장도 별미. 70여 년의 역사를 자랑한다.\n메뉴 \n짜장면(6천원), 짬뽕, 우동(각 8천원)

In [23]:
full_text = '\n'.join([pages_data[i]['text'] for i in range(len(pages_data))])

In [24]:
# PDF 파싱
print("\n📝 식당 정보 파싱 중...\n")
rag.parse_pdf_pages(full_text)

print(f"\n✅ 총 {len(rag.restaurants)}개 식당 파싱 완료\n")


📝 식당 정보 파싱 중...

✅ 파싱 완료: 포항돼지국밥
✅ 파싱 완료: 송정3대국밥
✅ 파싱 완료: 형제돼지국밥
✅ 파싱 완료: 60년전통할매국밥
✅ 파싱 완료: 서진섭돼지국밥
✅ 파싱 완료: 신창국밥
✅ 파싱 완료: 밀양국밥
✅ 파싱 완료: 자매국밥
✅ 파싱 완료: 내호냉면
✅ 파싱 완료: 대성밀냉면
✅ 파싱 완료: 개금밀면
✅ 파싱 완료: 할매가야밀면
✅ 파싱 완료: 영주동삼대복국집
✅ 파싱 완료: 할매복집
✅ 파싱 완료: 초원복국
✅ 파싱 완료: 새송도복국
✅ 파싱 완료: 금수복국
✅ 파싱 완료: 남해복국
✅ 파싱 완료: 박옥희할매집원조복국
✅ 파싱 완료: 영주복국
✅ 파싱 완료: 가마솥생복집
✅ 파싱 완료: 일광대복집
✅ 파싱 완료: 기와집대구탕
✅ 파싱 완료: 오뚜기식당
✅ 파싱 완료: 중앙식당
✅ 파싱 완료: 속씨원한대구탕
✅ 파싱 완료: 마가만두 馬家
✅ 파싱 완료: 신발원 新發園
✅ 파싱 완료: 옥생관 玉生館
✅ 파싱 완료: 동화반점 東華飯店
✅ 파싱 완료: 신흥관 新興館
✅ 파싱 완료: 신흥반점 新興飯店
✅ 파싱 완료: 삼생원 三生園
✅ 파싱 완료: 홍성방 鴻聖坊
✅ 파싱 완료: 대한명인기장곰장어
✅ 파싱 완료: 성일집
✅ 파싱 완료: 원조소문난산곰장어
✅ 파싱 완료: 온천입구기장곰장어
✅ 파싱 완료: 용암할매횟집
✅ 파싱 완료: 남항횟집
✅ 파싱 완료: 장군멸치회촌
✅ 파싱 완료: 부산명물횟집
✅ 파싱 완료: 꺼먹동네
✅ 파싱 완료: 제일장횟집
✅ 파싱 완료: 무진장횟집
✅ 파싱 완료: 영변횟집
✅ 파싱 완료: 용광횟집
✅ 파싱 완료: 궁중해물탕조씨집
✅ 파싱 완료: 원조바다집
✅ 파싱 완료: 산수갑산
✅ 파싱 완료: 삼대돼지불고기
✅ 파싱 완료: 은하갈비
✅ 파싱 완료: 구포촌국수
✅ 파싱 완료: 이원화구포국시
✅ 파싱 완료: 평산옥
✅ 파싱 완료: 동래할매파전
✅ 파싱 완료: 서울깍두기
✅ 파싱 완료: 18번완당집
✅ 파싱 완료: 원조18번완당발국수
✅ 파싱 완료: 급행장 急行莊
✅ 파싱 완료: 새진주식당
✅ 파싱 완료: 옛날오막집


In [25]:
data = [asdict(res) for res in rag.restaurants] 

df = pd.DataFrame(data)

In [26]:
# 검증 로직
existing_names = set(df['name_ko'].unique())
missing_restaurants = [name for name in restaurant_list if name not in existing_names]

print(missing_restaurants)

[]


In [27]:
# 데이터프레임의 각 행을 순회하며 상세 정보 출력
for _, row in df.iterrows():
    print(f"{row['name_ko']}")
    print(f"📄 설명: {row['description']}")
    print(f"🍴 메뉴: {row['menu']}")
    print(f"⏰ 영업: {row['hours']}")
    print(f"📍 주소: {row['address']}")
    print(f"📞 전화: {row['phone']}")
    print("-" * 50)  # 구분을 위한 긴 점선

포항돼지국밥
📄 설명: 서면시장 돼지국밥거리 쪽에 있는 80여 년 전통의 국밥집이다. 따로 나오는 면 사리를 국밥에넣어서 양념이 된 부추와 함께 먹는 맛이 일품이다. 옛날부터 한결같은 맛으로 많은 사람들이 찾는 곳이다.
🍴 메뉴: 돼지국밥, 내장국밥, 섞어국밥, 순대국밥, 따로국밥, 모둠국밥(각 9천원, 특 1만원), 수육백반(1만2천원) 순대한접시(1만3천원), 수육(소 3만원, 대 3만5천원)
⏰ 영업: 24시간 영업 - 연중무휴
📍 주소: 부산광역시 부산진구 서면로68번길 27 (부전동)
📞 전화: 051-807-5439
--------------------------------------------------
송정3대국밥
📄 설명: 돼지의 좋은 부위를 사용해 돼지국밥의 국물이뽀얗고 깔끔하다. 당면 대신 찹쌀을 넣어서 만드는 찹쌀순대도 인기가 좋다. 서면의 돼지국밥 골목의 원조집 중 하나로, 80여 년의 역사를 자랑한다.
🍴 메뉴: 돼지국밥, 순대국밥, 내장국밥, 따로국밥(각 9천원), 수육백반(1만1천원), 찹쌀순대(소 1만원, 대 1만3천원), 수육(소 2만8천원, 대 3만3천원)
⏰ 영업: 24시간 영업 - 명절 휴무
📍 주소: 부산광역시 부산진구 서면로68번길 33 (부전동)
📞 전화: 051-806-5722
--------------------------------------------------
형제돼지국밥
📄 설명: 사골 육수에 15가지 천연 재료를 첨가하여 푹 끓인 돼지국밥의 맛이 좋다. 국밥을 주문하면 푸짐한 양의 수육을 맛볼 수 있으며, 소량의 수육과국밥이 함께 나오는 수육백반도 인기가 많다. 하이브리드 국밥은 돼지의 특수 부위로 만든 국밥으로, 하루에 20개로 한정판매된다.
🍴 메뉴: 불꽃국밥, 콩나물해장국밥, 하이브리드국밥(각 1만원), 돼지구리국밥, 마라국밥(각 1만1천원), 맑은국밥, 뽀얀국밥(각 1만원), 보쌈백반(1만4천원)
⏰ 영업: 08:00~22:00(마지막 주문 21:00) - 명절 당일 휴무
📍 주소: 부산광역시 남구